# Load CBS Netherlands (Statistics Netherlands) Metadata

This notebook fetches statistical table metadata from CBS Netherlands - covering Dutch economic, demographic, labor, health, and social statistics.

In [ ]:
%pip install cbsodata tqdm --quiet

In [ ]:
# Configuration
CATALOG = "main_catalog"
SCHEMA = "dev"
TABLE_NAME = "cbs_indicators"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

FRESH_START = False

In [ ]:
import cbsodata
from tqdm import tqdm
from pyspark.sql.types import StructType, StructField, StringType

In [ ]:
schema = StructType([
    StructField("indicator_id", StringType(), False),
    StructField("indicator_name", StringType(), True),
    StructField("long_definition", StringType(), True),
    StructField("source_organization", StringType(), True),
    StructField("source", StringType(), True),
    StructField("topics", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("periodicity", StringType(), True),
    StructField("aggregation_method", StringType(), True),
    StructField("license_type", StringType(), True),
    StructField("embedding_text", StringType(), True),
])

In [ ]:
# Get existing indicator IDs or create table
existing_ids = set()

if FRESH_START:
    spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME}")
    print("Fresh start - dropped existing table")
else:
    try:
        existing_df = spark.sql(f"SELECT indicator_id FROM {FULL_TABLE_NAME}")
        existing_ids = set(row.indicator_id for row in existing_df.collect())
        print(f"Resuming - found {len(existing_ids)} existing records")
    except:
        print("Table doesn't exist yet, starting fresh")

if not existing_ids or FRESH_START:
    empty_df = spark.createDataFrame([], schema)
    empty_df.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("overwrite") \
        .saveAsTable(FULL_TABLE_NAME)
    
    # Set table description
    spark.sql(f"""
        COMMENT ON TABLE {FULL_TABLE_NAME} IS 
        'CBS Netherlands (Statistics Netherlands) table metadata covering Dutch economic, demographic, labor, health, and social statistics.'
    """)
    print(f"Created table {FULL_TABLE_NAME} with CDF enabled")

In [ ]:
# Fetch all CBS tables
print("Fetching CBS table list...")
all_tables = cbsodata.get_table_list()

# Filter out already loaded
tables = [t for t in all_tables if t.get("Identifier") not in existing_ids]

print(f"Total CBS tables: {len(all_tables)}")
print(f"Already loaded: {len(existing_ids)}")
print(f"Remaining to load: {len(tables)}")

In [ ]:
# Load tables
for table in tqdm(tables, desc="Loading CBS tables"):
    try:
        indicator_id = table.get("Identifier", "")
        indicator_name = table.get("Title", "") or ""
        
        # Build long definition from available fields
        short_desc = table.get("ShortDescription", "") or ""
        summary = table.get("Summary", "") or ""
        long_definition = f"{short_desc} {summary}".strip()
        
        # Extract frequency/periodicity
        frequency = table.get("Frequency", "") or ""
        
        # Extract catalog/theme as topic
        catalog = table.get("Catalog", "") or "CBS"
        
        # Create embedding text
        embedding_text = f"{indicator_name}. {long_definition}".strip()
        if not embedding_text or embedding_text == ".":
            embedding_text = indicator_name or indicator_id

        record = [(
            indicator_id,
            indicator_name,
            long_definition[:5000] if len(long_definition) > 5000 else long_definition,
            "Statistics Netherlands (CBS)",
            "CBS StatLine",
            catalog,
            "",
            frequency,
            "",
            "CC BY 4.0",
            embedding_text[:5000] if len(embedding_text) > 5000 else embedding_text,
        )]

        row_df = spark.createDataFrame(record, schema)
        row_df.write.format("delta").mode("append").saveAsTable(FULL_TABLE_NAME)

    except Exception as e:
        print(f"Error loading {indicator_id}: {e}")
        continue

print("Done!")

In [ ]:
# Verify
count = spark.sql(f"SELECT COUNT(*) FROM {FULL_TABLE_NAME}").collect()[0][0]
print(f"Total CBS tables in table: {count}")
display(spark.sql(f"SELECT * FROM {FULL_TABLE_NAME} LIMIT 5"))